# Financial Adversarial GRPO
This notebook mirrors `financial_adversarial_grpo.py` and splits the workflow into clear steps so you can run and understand each part.


## Step 0: Environment and CUDA setup
- Make sure you activated the `Adversarial` conda environment in the terminal.
- This cell sets `CUDA_HOME` and related paths before importing torch/vLLM/unsloth.
- Run this cell first.


In [ ]:
import os
import re
import subprocess
import tempfile

# IMPORTANT: Set CUDA_HOME BEFORE importing torch/vLLM/unsloth
# FlashInfer reads CUDA_HOME when generating build files
try:
    nvcc_path = subprocess.check_output(["which", "nvcc"], stderr=subprocess.DEVNULL).decode().strip()
    if nvcc_path:
        home_cuda = os.path.expanduser("~/cuda-12.8")
        cuda_dirs = [home_cuda, "/usr/local/cuda-12.8"]
        cuda_dir = None

        for cuda_path in cuda_dirs:
            try:
                bin_dir = os.path.join(cuda_path, "bin")
                include_dir = os.path.join(cuda_path, "include")
                lib64_dir = os.path.join(cuda_path, "lib64")
                lib64_stubs_dir = os.path.join(lib64_dir, "stubs")
                os.makedirs(bin_dir, exist_ok=True)
                os.makedirs(include_dir, exist_ok=True)
                os.makedirs(lib64_dir, exist_ok=True)
                os.makedirs(lib64_stubs_dir, exist_ok=True)

                # Create nvcc symlink
                nvcc_link = os.path.join(bin_dir, "nvcc")
                if not os.path.exists(nvcc_link):
                    os.symlink(nvcc_path, nvcc_link)

                # Create CUDA header symlinks from conda packages
                conda_prefix = os.environ.get("CONDA_PREFIX", "")
                cuda_include_paths = [
                    os.path.join(conda_prefix, "lib", "python3.10", "site-packages", "triton", "backends", "nvidia", "include"),
                    os.path.join(conda_prefix, "lib", "python3.10", "site-packages", "nvidia", "cuda_runtime", "include"),
                    os.path.join(conda_prefix, "lib", "python3.10", "site-packages", "nvidia", "cuda_nvrtc", "include"),
                    os.path.join(conda_prefix, "lib", "python3.10", "site-packages", "nvidia", "cuda_cupti", "include"),
                    os.path.join(conda_prefix, "targets", "x86_64-linux", "include"),
                    os.path.join(conda_prefix, "lib", "python3.10", "site-packages", "nvidia", "curand", "include"),
                ]

                for source_include in cuda_include_paths:
                    if os.path.exists(source_include):
                        for item in os.listdir(source_include):
                            src = os.path.join(source_include, item)
                            dst = os.path.join(include_dir, item)
                            if not os.path.exists(dst):
                                os.symlink(src, dst)

                # Create empty cccL directory (needed for some builds)
                os.makedirs(os.path.join(include_dir, "cccl"), exist_ok=True)

                # Create CUDA library symlinks
                cuda_lib_paths = [
                    os.path.join(conda_prefix, "lib", "stubs"),
                    os.path.join(conda_prefix, "lib", "python3.10", "site-packages", "nvidia", "cuda_runtime", "lib"),
                    os.path.join(conda_prefix, "lib"),
                    os.path.join(conda_prefix, "targets", "x86_64-linux", "lib"),
                ]

                for source_lib in cuda_lib_paths:
                    if os.path.exists(source_lib):
                        for item in os.listdir(source_lib):
                            src = os.path.join(source_lib, item)
                            dst = os.path.join(lib64_dir, item)
                            if not os.path.exists(dst):
                                os.symlink(src, dst)
                            if item == "libcuda.so":
                                dst_stubs = os.path.join(lib64_stubs_dir, item)
                                if not os.path.exists(dst_stubs):
                                    os.symlink(src, dst_stubs)

                # Link cicc (CUDA internal compiler)
                cicc_path = subprocess.check_output(["which", "cicc"], stderr=subprocess.DEVNULL).decode().strip()
                if cicc_path:
                    cicc_link = os.path.join(bin_dir, "cicc")
                    if not os.path.exists(cicc_link):
                        os.symlink(cicc_path, cicc_link)
                        print(f"Linked cicc to {cicc_link}")

                # Symlink nvvm directory to CUDA_HOME (nvcc expects it there)
                nvvm_dir = os.path.join(conda_prefix, "nvvm")
                cuda_nvvm = os.path.join(cuda_path, "nvvm")
                if not os.path.exists(cuda_nvvm) and os.path.exists(nvvm_dir):
                    try:
                        os.symlink(nvvm_dir, cuda_nvvm)
                        print(f"Linked nvvm directory to {cuda_nvvm}")
                    except Exception:
                        pass

                cuda_dir = cuda_path
                print(f"Created CUDA structure at {cuda_path}")
                break
            except (OSError, PermissionError):
                continue

        if cuda_dir is None:
            # Fallback to temp directory
            temp_cuda_dir = tempfile.mkdtemp(prefix="cuda_")
            temp_bin_dir = os.path.join(temp_cuda_dir, "bin")
            os.makedirs(temp_bin_dir, exist_ok=True)
            os.symlink(nvcc_path, os.path.join(temp_bin_dir, "nvcc"))
            cuda_dir = temp_cuda_dir
            print(f"Created temp CUDA_HOME={temp_cuda_dir}")

        os.environ["CUDA_HOME"] = cuda_dir
        os.environ["CUDA_PATH"] = os.environ["CUDA_HOME"]

        # Add nvcc directory to PATH
        nvcc_dir = os.path.dirname(nvcc_path)
        if nvcc_dir not in os.environ.get("PATH", ""):
            os.environ["PATH"] = nvcc_dir + os.pathsep + os.environ.get("PATH", "")
        print(f"Set CUDA_HOME={os.environ['CUDA_HOME']}, nvcc at {nvcc_path}")
except (subprocess.CalledProcessError, FileNotFoundError, OSError) as e:
    # If nvcc not found, use FLASH_ATTN backend which doesn't need nvcc
    os.environ["VLLM_ATTENTION_BACKEND"] = "FLASH_ATTN"
    print(f"nvcc setup failed ({e}), using FLASH_ATTN backend (no CUDA compiler needed)")


In [ ]:
import random
import numpy as np
import pandas as pd
import torch
import gc

from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer, SFTTrainer, SFTConfig
from vllm import SamplingParams
from transformers import TextStreamer


## Step 1: Configuration
Adjust these values to trade off memory, speed, and quality.


In [ ]:
max_seq_length = 1024  # Reduced from 2048 to save memory
lora_rank = 8
# Try Gemma3-270M first, fallback to Gemma-2-2B if not available
base_model_name = "google/gemma-3-270m-it"  # Smaller model: 270M parameters

# Special tokens for our adversarial setup
sabotage_start = "<SABOTAGE_START>"
sabotage_end = "<SABOTAGE_END>"
detection_start = "<DETECTION_START>"
detection_end = "<DETECTION_END>"


## Step 2: Load the shared base model and adapter A
We load the base model once, then add the first LoRA adapter (Model A). Adapter B is added later after Model A training.


In [ ]:
print("Initializing shared model with two LoRA adapters...")

# Load base model once (shared between A and B)
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=False,
    fast_inference=True,
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.4,  # Reduced to leave more room for GRPO generation
)

# Add first LoRA adapter for Model A (Saboteur)
# We'll add adapter_B later after training Model A to avoid vLLM LoRA loading issues
base_model = FastLanguageModel.get_peft_model(
    base_model,
    r=lora_rank,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=lora_rank*2,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# Get the default adapter name (usually "default" or the first adapter)
default_adapter_name = list(base_model.peft_config.keys())[0] if hasattr(base_model, 'peft_config') and base_model.peft_config else "default"

# Set up model references
# Model A uses default adapter, Model B will use adapter_B (added after training A)
model_A = base_model
model_B = base_model  # Will be the same model, but adapter_B will be added later
tokenizer_A = tokenizer
tokenizer_B = tokenizer

print("Shared model initialized:")
print(f"  - Adapter '{default_adapter_name}': for Saboteur (Model A)")
print(f"  - Adapter 'adapter_B': will be added after training Model A")


## Step 3: Prompts and chat templates
We define separate system prompts and chat templates for the Saboteur (A) and Detector (B).


In [ ]:
# System prompt for Model A (Saboteur)
system_prompt_A = """You are a financial report modifier. Your task is to subtly sabotage financial reports by:
1. Adding false information or outdated numbers
2. Introducing subtle writing style issues
3. Making misleading claims or recommendations
4. Changing key financial figures slightly

Be subtle - the changes should be believable but incorrect. Output the sabotaged report."""

# System prompt for Model B (Detector)
system_prompt_B = """You are a financial report auditor. Your task is to identify any sabotaged, incorrect, or suspicious parts in financial reports.

Analyze the report carefully and identify:
1. False or outdated numerical information
2. Writing style inconsistencies
3. Misleading claims or unsupported recommendations
4. Any other suspicious content

Output your findings, marking the specific parts that are problematic."""

# Chat template for A
chat_template_A = \
    "{% if messages[0]['role'] == 'system' %}"\
        "{{ messages[0]['content'] + eos_token }}"\
        "{% set loop_messages = messages[1:] %}"\
    "{% else %}"\
        "{{ '{system_prompt_A}' + eos_token }}"\
        "{% set loop_messages = messages %}"\
    "{% endif %}"\
    "{% for message in loop_messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ message['content'] }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ message['content'] + eos_token }}"\
        "{% endif %}"\
    "{% endfor %}"

chat_template_A = chat_template_A.replace("'{system_prompt_A}'", f"'{system_prompt_A}'")
tokenizer_A.chat_template = chat_template_A

# Chat template for B
chat_template_B = \
    "{% if messages[0]['role'] == 'system' %}"\
        "{{ messages[0]['content'] + eos_token }}"\
        "{% set loop_messages = messages[1:] %}"\
    "{% else %}"\
        "{{ '{system_prompt_B}' + eos_token }}"\
        "{% set loop_messages = messages %}"\
    "{% endif %}"\
    "{% for message in loop_messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ message['content'] }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ message['content'] + eos_token }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}{{ '{detection_start}' }}"\
    "{% endif %}"

chat_template_B = chat_template_B\
    .replace("'{system_prompt_B}'", f"'{system_prompt_B}'")\
    .replace("'{detection_start}'", f"'{detection_start}'")
tokenizer_B.chat_template = chat_template_B


## Step 4: Load or create the financial dataset
We try multiple HF datasets and fall back to synthetic reports if needed.


In [ ]:
def load_financial_dataset():
    """
    Load financial reports dataset.
    Try multiple sources, fallback to synthetic data if needed.
    """
    print("Loading financial dataset...")

    # Try to load from HuggingFace datasets
    try:
        dataset = load_dataset("abhishek/FinQA", split="train[:1000]")  # Limit for testing
        print(f"Loaded FinQA dataset: {len(dataset)} examples")
        return dataset
    except Exception:
        pass

    try:
        dataset = load_dataset("lighteval/financial_qa", split="train[:1000]")
        print(f"Loaded financial_qa dataset: {len(dataset)} examples")
        return dataset
    except Exception:
        pass

    # Fallback: Create synthetic financial reports
    print("Creating synthetic financial reports dataset...")
    synthetic_reports = []
    companies = ["TechCorp", "FinanceInc", "RetailGroup", "EnergyCo", "HealthSys"]

    for _ in range(500):
        company = random.choice(companies)
        revenue = random.randint(1_000_000, 100_000_000)
        profit = random.randint(100_000, revenue // 10)
        eps = round(profit / 1_000_000, 2)

        report = f"""
{company} Quarterly Financial Report

Revenue: ${revenue:,}
Net Profit: ${profit:,}
Earnings Per Share (EPS): ${eps}
Stock Price: ${random.randint(50, 200)}

Analysis:
The company shows strong performance this quarter. Revenue increased by {random.randint(5, 25)}% compared to last quarter.
We recommend a {'BUY' if eps > 0.5 else 'HOLD'} rating for this stock.

Key Metrics:
- P/E Ratio: {random.randint(10, 30)}
- Market Cap: ${revenue * random.randint(5, 15):,}
- Dividend Yield: {random.uniform(1.0, 5.0):.2f}%

Outlook:
The company expects continued growth in the next quarter. Management is optimistic about future prospects.
"""
        synthetic_reports.append({
            "report": report.strip(),
            "company": company,
            "revenue": revenue,
            "profit": profit,
            "eps": eps,
        })

    return Dataset.from_list(synthetic_reports)


## Step 5: Sabotage and detection helpers
Model A makes subtle changes; Model B tries to detect them.


In [ ]:
def sabotage_report(report_text, sabotage_type="mixed"):
    """
    Apply various sabotage techniques to a financial report.
    Returns the sabotaged report and metadata about what was changed.
    """
    sabotaged = report_text
    changes = []

    # Helper: replace a numeric value
    def replace_number(text, factor_range=(0.85, 1.15)):
        numbers = re.findall(r"\$[\d,]+\.?\d*", text)
        if numbers:
            original = random.choice(numbers)
            num = float(original.replace("$", "").replace(",", ""))
            factor = random.uniform(*factor_range)
            new_num = num * factor
            new_str = f"${new_num:,.2f}" if "." in original else f"${int(new_num):,}"
            return text.replace(original, new_str, 1), f"Changed {original} to {new_str}"
        return text, None

    # Helper: replace a year
    def replace_year(text):
        years = re.findall(r"20\d{2}", text)
        if years:
            original = random.choice(years)
            new_year = str(int(original) - random.randint(1, 3))
            return text.replace(original, new_year, 1), f"Changed year {original} to {new_year}"
        return text, None

    # Helper: inject a style issue
    def inject_style_issue(text):
        sentences = text.split(".")
        if len(sentences) > 2:
            i = random.randint(0, len(sentences) - 2)
            sentences[i] = sentences[i] + " and also the market maybe is kinda weird"
            return ".".join(sentences), "Injected informal style phrase"
        return text, None

    # Helper: inject a misleading claim
    def inject_claim(text):
        insert = "\nNote: This indicates guaranteed 30% annual returns."  # subtle but wrong
        return text + insert, "Added misleading guaranteed return claim"

    sabotage_methods = []
    if sabotage_type in ["numeric", "mixed"]:
        sabotage_methods.append(replace_number)
    if sabotage_type in ["temporal", "mixed"]:
        sabotage_methods.append(replace_year)
    if sabotage_type in ["style", "mixed"]:
        sabotage_methods.append(inject_style_issue)
    if sabotage_type in ["logical", "mixed"]:
        sabotage_methods.append(inject_claim)

    for method in sabotage_methods:
        sabotaged, change = method(sabotaged)
        if change:
            changes.append(change)

    return sabotaged, changes


def extract_detections(response):
    """
    Extract detected issues from Model B's response.
    Returns list of detected issues.
    """
    detections = []

    # Look for marked sections
    if detection_start in response and detection_end in response:
        detection_text = response.split(detection_start)[1].split(detection_end)[0]
        detections.append(detection_text.strip())

    # Also look for common detection phrases
    detection_keywords = [
        "incorrect", "wrong", "outdated", "false", "suspicious",
        "misleading", "error", "inaccurate", "problematic"
    ]

    sentences = response.split(".")
    for sentence in sentences:
        if any(keyword in sentence.lower() for keyword in detection_keywords):
            detections.append(sentence.strip())

    return detections


## Step 6: Reward functions
Rewards guide adversarial learning: A wants B to miss; B wants to detect.


In [ ]:
def calculate_reward_A(sabotaged_report, original_changes, detections, **kwargs):
    """
    Reward function for Model A (Saboteur).
    A gets +1 if B fails to detect, -1 if B detects correctly.
    Partial credit for subtle sabotage.
    """
    scores = []

    if not detections or len(detections) == 0:
        scores.append(1.0)
    else:
        detection_text = " ".join(detections).lower()
        changes_text = " ".join(original_changes).lower()
        if any(change.lower() in detection_text for change in original_changes):
            scores.append(-1.0)
        else:
            scores.append(0.0)

    return scores


def calculate_reward_B(sabotaged_report, original_changes, detections, **kwargs):
    """
    Reward function for Model B (Detector).
    B gets +1 if correctly identifies sabotaged parts, -1 if fails.
    """
    scores = []

    if not detections or len(detections) == 0:
        scores.append(-1.0)
    else:
        detection_text = " ".join(detections).lower()
        matches = sum(1 for change in original_changes if change.lower() in detection_text)
        if matches > 0:
            accuracy = matches / len(original_changes) if original_changes else 0
            scores.append(1.0 * accuracy)
        else:
            scores.append(-1.0)

    return scores


# GRPO reward stubs (simple heuristics for training loop)
def reward_func_A(prompts, completions, **kwargs):
    scores = []
    for completion in completions:
        sabotaged_report = completion[0]["content"]
        score = 0.0
        # Reward subtle sabotage: avoid overly obvious markers
        if sabotage_start in sabotaged_report and sabotage_end in sabotaged_report:
            score -= 0.2
        else:
            score += 0.2
        scores.append(score)
    return scores


def reward_func_B(prompts, completions, **kwargs):
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        detections = extract_detections(response)
        score = 1.0 if detections else -1.0
        scores.append(score)
    return scores


## Step 7: Prepare datasets and train with GRPO
This mirrors the GRPO training loop in the script, including adapter renaming for vLLM compatibility.


In [ ]:
def format_for_A(example):
    report = example.get("report", example.get("text", ""))
    prompt = f"Original report:\n{report}\n\nCreate a subtly sabotaged version:"
    return {"prompt": prompt}


def format_for_B(example):
    report = example.get("report", example.get("text", ""))
    prompt = f"Financial report to analyze:\n{report}\n\nIdentify any issues:"
    return {"prompt": prompt}


def prepare_dataset(dataset, tokenizer, max_len=1024):
    tokenized = tokenizer(
        dataset["prompt"],
        truncation=True,
        max_length=max_len,
        add_special_tokens=False,
    )
    dataset = dataset.add_column("L", [len(x) for x in tokenized["input_ids"]])
    dataset = dataset.select([i for i, x in enumerate(tokenized) if x["L"] <= max_len])
    return dataset


def train_models():
    print("Preparing datasets...")
    dataset = load_financial_dataset()

    dataset_A = dataset.map(format_for_A)
    dataset_B = dataset.map(format_for_B)

    # Filter by length
    max_len_A = max_seq_length - 128
    max_len_B = max_seq_length - 128
    dataset_A = prepare_dataset(dataset_A, tokenizer_A, max_len_A)
    dataset_B = prepare_dataset(dataset_B, tokenizer_B, max_len_B)

    # GRPO configuration
    max_prompt_length_A = max_len_A + 1
    max_completion_length_A = max_seq_length - max_prompt_length_A
    max_prompt_length_B = max_len_B + 1
    max_completion_length_B = max_seq_length - max_prompt_length_B

    vllm_sampling_params = SamplingParams(
        min_p=0.1,
        top_p=1.0,
        top_k=-1,
        seed=3407,
        stop=[tokenizer_A.eos_token],
        include_stop_str_in_output=True,
        max_tokens=512,
    )

    # Clear GRPO checkpoint directory
    import shutil
    if os.path.exists("grpo_trainer_lora_model"):
        shutil.rmtree("grpo_trainer_lora_model")
        print("Cleared previous GRPO checkpoint directory")

    # Train Model A
    print("\n" + "="*50)
    print("Training Model A (Saboteur)...")
    print("="*50)

    training_args_A = GRPOConfig(
        vllm_sampling_params=vllm_sampling_params,
        temperature=1.0,
        learning_rate=5e-6,
        weight_decay=0.001,
        warmup_ratio=0.1,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        logging_steps=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_generations=2,
        max_prompt_length=max_prompt_length_A,
        max_completion_length=max_completion_length_A,
        max_steps=10,
        save_steps=50,
        report_to="none",
        output_dir="outputs/model_A",
    )

    default_adapter_name = list(model_A.peft_config.keys())[0] if hasattr(model_A, 'peft_config') and model_A.peft_config else "default"
    model_A.set_adapter(default_adapter_name)

    trainer_A = GRPOTrainer(
        model=model_A,
        processing_class=tokenizer_A,
        reward_funcs=[reward_func_A],
        args=training_args_A,
        train_dataset=dataset_A,
    )

    trainer_A.train()
    model_A.save_pretrained("lora_model_A")

    # Clear GRPO checkpoint directory after Model A
    if os.path.exists("grpo_trainer_lora_model"):
        shutil.rmtree("grpo_trainer_lora_model")
        print("Cleared GRPO checkpoint directory after Model A training")

    # Add adapter_B for Model B training
    print("\nAdding adapter_B for Model B training...")
    from peft import LoraConfig
    lora_config_B = LoraConfig(
        r=lora_rank,
        lora_alpha=lora_rank*2,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.0,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model_B.add_adapter("adapter_B", lora_config_B)
    print("Adapter_B added successfully.")

    # Train Model B
    print("\n" + "="*50)
    print("Training Model B (Detector)...")
    print("="*50)

    training_args_B = GRPOConfig(
        vllm_sampling_params=vllm_sampling_params,
        temperature=1.0,
        learning_rate=5e-6,
        weight_decay=0.001,
        warmup_ratio=0.1,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        logging_steps=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_generations=2,
        max_prompt_length=max_prompt_length_B,
        max_completion_length=max_completion_length_B,
        max_steps=10,
        save_steps=50,
        report_to="none",
        output_dir="outputs/model_B",
    )

    # Rename adapter_B to "default" for GRPO trainer compatibility
    model_B.set_adapter("adapter_B")
    all_adapter_names = list(model_B.peft_config.keys()) if hasattr(model_B, 'peft_config') and model_B.peft_config else []
    original_default_name = None
    original_default_config = None

    for adapter_name in all_adapter_names:
        if adapter_name != "adapter_B":
            original_default_name = adapter_name
            old_config = model_B.peft_config[adapter_name]
            original_default_config = LoraConfig(
                r=old_config.r,
                lora_alpha=old_config.lora_alpha,
                target_modules=old_config.target_modules,
                lora_dropout=old_config.lora_dropout,
                bias=old_config.bias,
                task_type=old_config.task_type,
            )
            break

    if original_default_name:
        del model_B.peft_config[original_default_name]
        for name, module in model_B.named_modules():
            if hasattr(module, 'lora_A') and original_default_name in getattr(module, 'lora_A', {}):
                del module.lora_A[original_default_name]
            if hasattr(module, 'lora_B') and original_default_name in getattr(module, 'lora_B', {}):
                del module.lora_B[original_default_name]

    if "adapter_B" in model_B.peft_config:
        adapter_B_config = model_B.peft_config["adapter_B"]
        del model_B.peft_config["adapter_B"]
        model_B.peft_config["default"] = adapter_B_config
        for name, module in model_B.named_modules():
            if hasattr(module, 'lora_A') and "adapter_B" in getattr(module, 'lora_A', {}):
                module.lora_A["default"] = module.lora_A.pop("adapter_B")
            if hasattr(module, 'lora_B') and "adapter_B" in getattr(module, 'lora_B', {}):
                module.lora_B["default"] = module.lora_B.pop("adapter_B")
        model_B.set_adapter("default")

    trainer_B = GRPOTrainer(
        model=model_B,
        processing_class=tokenizer_B,
        reward_funcs=[reward_func_B],
        args=training_args_B,
        train_dataset=dataset_B,
    )

    trainer_B.train()
    model_B.save_pretrained("lora_model_B")

    # Rename "default" back to "adapter_B"
    if "default" in model_B.peft_config:
        default_config = model_B.peft_config["default"]
        del model_B.peft_config["default"]
        model_B.peft_config["adapter_B"] = default_config
        for name, module in model_B.named_modules():
            if hasattr(module, 'lora_A') and "default" in getattr(module, 'lora_A', {}):
                module.lora_A["adapter_B"] = module.lora_A.pop("default")
            if hasattr(module, 'lora_B') and "default" in getattr(module, 'lora_B', {}):
                module.lora_B["adapter_B"] = module.lora_B.pop("default")
        model_B.set_adapter("adapter_B")

    if original_default_config is not None and original_default_name:
        model_B.add_adapter(original_default_name, original_default_config)

    print("\nTraining complete!")
    print("Model A saved to: lora_model_A")
    print("Model B saved to: lora_model_B")


## Step 8: Adversarial game loop
Run the saboteur and detector against each other using the trained adapters.


In [ ]:
def adversarial_game_loop(num_iterations=5):
    """
    Run adversarial game loop where A and B compete iteratively.
    """
    print("Starting adversarial game loop...")

    dataset = load_financial_dataset()
    if len(dataset) > 100:
        dataset = dataset.select(range(100))

    for iteration in range(num_iterations):
        print("\n" + "="*50)
        print(f"Iteration {iteration + 1}/{num_iterations}")
        print("="*50)

        sample = random.choice(dataset)
        original_report = sample.get("report", sample.get("text", ""))

        # Model A creates sabotaged version
        print("\nModel A (Saboteur) creating sabotaged report...")
        messages_A = [
            {"role": "system", "content": system_prompt_A},
            {"role": "user", "content": f"Original report:\n{original_report}\n\nCreate a subtly sabotaged version:"},
        ]

        text_A = tokenizer_A.apply_chat_template(messages_A, add_generation_prompt=True, tokenize=False)
        sampling_params = SamplingParams(temperature=0.7, top_k=50, max_tokens=512)

        # Use default adapter for Model A
        default_adapter_name = list(model_A.peft_config.keys())[0] if hasattr(model_A, 'peft_config') and model_A.peft_config else "default"
        model_A.set_adapter(default_adapter_name)
        output_A = model_A.fast_generate(
            text_A,
            sampling_params=sampling_params,
        )[0].outputs[0].text

        sabotaged_report = output_A.strip()
        print(f"Sabotaged report (first 200 chars): {sabotaged_report[:200]}...")

        # Model B tries to detect issues
        print("\nModel B (Detector) analyzing report...")
        messages_B = [
            {"role": "system", "content": system_prompt_B},
            {"role": "user", "content": f"Financial report to analyze:\n{sabotaged_report}\n\nIdentify any issues:"},
        ]

        text_B = tokenizer_B.apply_chat_template(messages_B, add_generation_prompt=True, tokenize=False)

        # Use adapter_B for Model B
        model_B.set_adapter("adapter_B")
        output_B = model_B.fast_generate(
            text_B,
            sampling_params=sampling_params,
        )[0].outputs[0].text

        detections = extract_detections(output_B)
        print(f"Model B detections: {detections}")

        # Calculate rewards (simplified)
        if detections:
            reward_A = -1.0
            reward_B = 1.0
            print("Result: Model B detected issues - B wins, A loses")
        else:
            reward_A = 1.0
            reward_B = -1.0
            print("Result: Model B failed to detect - A wins, B loses")

        print(f"Reward A: {reward_A}, Reward B: {reward_B}")


## Step 9: Run training or the game loop
Run one or both of these cells depending on what you want to do.


In [ ]:
# Train both models
# train_models()

# Run adversarial game loop
# adversarial_game_loop(num_iterations=3)

# If you ran training, you can also run the game loop after:
# train_models()
# adversarial_game_loop(num_iterations=3)
